In [1]:
import os
import random
import warnings

import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [3]:
def read_indices(path):
    with open(path, "r", encoding="utf-8") as f:
        return [int(x.strip()) for x in f if x.strip()]

dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
df = dataset["train"].to_pandas()

train_idx = read_indices("data/train_idx.txt")
val_idx = read_indices("data/val_idx.txt")
test_idx = read_indices("data/test_idx.txt")

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

train_df["type"] = train_df["type"].fillna("Unknown")
val_df["type"] = val_df["type"].fillna("Unknown")
test_df["type"] = test_df["type"].fillna("Unknown")

print(train_df.shape, val_df.shape, test_df.shape)

(49412, 16) (6176, 16) (6177, 16)


In [4]:
def normalize_text_col(s):
    s = s.fillna("").astype(str)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def make_text(df):
    subject = normalize_text_col(df["subject"])
    body = normalize_text_col(df["body"])
    text = subject + " [SEP] " + body
    return text.str.strip()

train_df["text"] = make_text(train_df)
val_df["text"] = make_text(val_df)
test_df["text"] = make_text(test_df)

In [5]:
y_train_queue = train_df["queue"]
y_val_queue = val_df["queue"]
y_test_queue = test_df["queue"]

In [6]:
def run_queue_experiment(
    tfidf_params,
    lr_params,
    train_text,
    val_text,
    y_train,
    y_val,
):
    vectorizer = TfidfVectorizer(**tfidf_params)
    X_train = vectorizer.fit_transform(train_text)
    X_val = vectorizer.transform(val_text)

    model = LogisticRegression(**lr_params)
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)

    result = {
        "val_macro_f1": f1_score(y_val, val_pred, average="macro"),
        "val_acc": accuracy_score(y_val, val_pred),
        "n_features": X_train.shape[1],
    }
    return result, vectorizer, model

In [7]:
experiments = [
    {
        "name": "baseline_like",
        "tfidf": {
            "max_features": 50000,
            "ngram_range": (1, 2),
            "min_df": 5,
        },
        "lr": {
            "max_iter": 300,
            "n_jobs": -1,
            "random_state": SEED,
        },
    },
    {
        "name": "balanced_12",
        "tfidf": {
            "max_features": 50000,
            "ngram_range": (1, 2),
            "min_df": 3,
            "sublinear_tf": True,
            "strip_accents": "unicode",
        },
        "lr": {
            "max_iter": 300,
            "n_jobs": -1,
            "random_state": SEED,
            "class_weight": "balanced",
            "C": 1.0,
        },
    },
    {
        "name": "balanced_13",
        "tfidf": {
            "max_features": 80000,
            "ngram_range": (1, 3),
            "min_df": 3,
            "sublinear_tf": True,
            "strip_accents": "unicode",
        },
        "lr": {
            "max_iter": 300,
            "n_jobs": -1,
            "random_state": SEED,
            "class_weight": "balanced",
            "C": 1.0,
        },
    },
    {
        "name": "balanced_13_c2",
        "tfidf": {
            "max_features": 80000,
            "ngram_range": (1, 3),
            "min_df": 3,
            "sublinear_tf": True,
            "strip_accents": "unicode",
        },
        "lr": {
            "max_iter": 300,
            "n_jobs": -1,
            "random_state": SEED,
            "class_weight": "balanced",
            "C": 2.0,
        },
    },
    {
        "name": "balanced_13_c05",
        "tfidf": {
            "max_features": 80000,
            "ngram_range": (1, 3),
            "min_df": 5,
            "sublinear_tf": True,
            "strip_accents": "unicode",
        },
        "lr": {
            "max_iter": 300,
            "n_jobs": -1,
            "random_state": SEED,
            "class_weight": "balanced",
            "C": 0.5,
        },
    },
]

In [8]:
results = []
saved_objects = {}

for exp in experiments:
    result, vectorizer, model = run_queue_experiment(
        tfidf_params=exp["tfidf"],
        lr_params=exp["lr"],
        train_text=train_df["text"],
        val_text=val_df["text"],
        y_train=y_train_queue,
        y_val=y_val_queue,
    )

    row = {
        "name": exp["name"],
        **result,
    }
    results.append(row)
    saved_objects[exp["name"]] = {
        "vectorizer": vectorizer,
        "model": model,
        "config": exp,
    }

results_df = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)
results_df

,name,val_macro_f1,val_acc,n_features
3,balanced_13_c2,0.812374,0.530602,80000
1,balanced_12,0.806597,0.495304,50000
2,balanced_13,0.790157,0.497895,80000
4,balanced_13_c05,0.748936,0.464702,80000
0,baseline_like,0.743565,0.571891,50000


In [9]:
best_name = results_df.iloc[0]["name"]
best_name

'balanced_13_c2'

In [10]:
best_vectorizer = saved_objects[best_name]["vectorizer"]
best_model = saved_objects[best_name]["model"]

X_test_best = best_vectorizer.transform(test_df["text"])
test_pred_queue = best_model.predict(X_test_best)

test_macro_f1 = f1_score(y_test_queue, test_pred_queue, average="macro")
test_acc_queue = accuracy_score(y_test_queue, test_pred_queue)

print("Best config:", best_name)
print("Test Macro-F1 (queue):", test_macro_f1)
print("Test Accuracy (queue):", test_acc_queue)

Best config: balanced_13_c2
Test Macro-F1 (queue): 0.7965477853438875
Test Accuracy (queue): 0.5337542496357455


In [11]:
X_train_best = best_vectorizer.transform(train_df["text"])
X_val_best = best_vectorizer.transform(val_df["text"])
X_test_best = best_vectorizer.transform(test_df["text"])

In [12]:
y_train_priority = train_df["priority"]
y_test_priority = test_df["priority"]

model_priority = LogisticRegression(
    max_iter=300,
    n_jobs=-1,
    random_state=SEED,
    C=1.0,
)

model_priority.fit(X_train_best, y_train_priority)
test_pred_priority = model_priority.predict(X_test_best)

priority_acc = accuracy_score(y_test_priority, test_pred_priority)
print("Priority Accuracy:", priority_acc)

Priority Accuracy: 0.6350979439857536


In [13]:
y_train_type = train_df["type"]
y_test_type = test_df["type"]

model_type = LogisticRegression(
    max_iter=300,
    n_jobs=-1,
    random_state=SEED,
    C=1.0,
)

model_type.fit(X_train_best, y_train_type)
test_pred_type = model_type.predict(X_test_best)

type_acc = accuracy_score(y_test_type, test_pred_type)
print("Type Accuracy:", type_acc)

Type Accuracy: 0.8766391452161243


In [14]:
final_score = (
    0.70 * test_macro_f1
    + 0.15 * priority_acc
    + 0.15 * type_acc
)

print("Final Score:", final_score)

Final Score: 0.7843440131210028
